In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, calinski_harabasz_score

# Configuración visual
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 12

In [4]:
# Cargar los datos ortogonalizados de la muestra 24
file_path = 'data/raw/intensidades_cobre.csv'
df_ = pd.read_csv(file_path)

df = df_.drop_duplicates(subset=['time'], keep='first').copy()
df

,date,time,cu,fe,mo,sc,zn
0,2025-07-13 15:00:00,35439.187500,38549.222656,1390.219971,1627.538940,182.446930,NaN
8,2025-07-13 17:00:00,38724.382812,39977.464844,1588.490601,1650.043213,202.994522,NaN
11,2025-07-13 17:45:00,38283.027344,40565.593750,1565.089844,1629.467163,192.615814,NaN
12,2025-07-13 18:00:00,37491.843750,40812.937500,1620.471558,1608.471313,196.744598,NaN
14,2025-07-13 18:30:00,38249.019531,40602.976562,1530.269409,1622.135254,200.647537,NaN
...,...,...,...,...,...,...,...
34182,2026-07-04 16:30:00,28620.107422,24066.855469,1245.356079,1466.279663,165.511337,NaN
34183,2026-07-04 16:45:00,29156.281250,24669.281250,1238.343750,1451.283325,162.046875,NaN
34184,2026-07-04 17:00:00,28331.830078,24118.513672,1347.781372,1460.660645,164.102859,NaN
34186,2026-07-04 17:30:00,29061.564453,24332.291016,1359.928833,1464.381836,167.200928,NaN


In [5]:
feature = ['fe', 'cu', 'mo', 'sc']
x = df[feature].dropna()
x

,fe,cu,mo,sc
0,1390.219971,38549.222656,1627.538940,182.446930
8,1588.490601,39977.464844,1650.043213,202.994522
11,1565.089844,40565.593750,1629.467163,192.615814
12,1620.471558,40812.937500,1608.471313,196.744598
14,1530.269409,40602.976562,1622.135254,200.647537
...,...,...,...,...
34182,1245.356079,24066.855469,1466.279663,165.511337
34183,1238.343750,24669.281250,1451.283325,162.046875
34184,1347.781372,24118.513672,1460.660645,164.102859
34186,1359.928833,24332.291016,1464.381836,167.200928


In [6]:
from sklearn.ensemble import IsolationForest
model = IsolationForest(
    n_estimators=100,
    contamination=0.02,
    random_state=42
)
model.fit(x)

,"n_estimators n_estimators: int, default=100The number of base estimators in the ensemble.",100
,"max_samples max_samples: ""auto"", int or float, default=""auto""The number of samples to draw from X to train each base estimator.- If int, then draw `max_samples` samples.- If float, then draw `max_samples * X.shape[0]` samples.- If ""auto"", then `max_samples=min(256, n_samples)`.If max_samples is larger than the number of samples provided,all samples will be used for all trees (no sampling).",'auto'
,"contamination contamination: 'auto' or float, default='auto'The amount of contamination of the data set, i.e. the proportionof outliers in the data set. Used when fitting to define the thresholdon the scores of the samples.- If 'auto', the threshold is determined as in the original paper.- If float, the contamination should be in the range (0, 0.5]... versionchanged:: 0.22 The default value of ``contamination`` changed from 0.1 to ``'auto'``.",0.02
,"max_features max_features: int or float, default=1.0The number of features to draw from X to train each base estimator.- If int, then draw `max_features` features.- If float, then draw `max(1, int(max_features * n_features_in_))` features.Note: using a float number less than 1.0 or integer less than number offeatures will enable feature subsampling and leads to a longer runtime.",1.0
,"bootstrap bootstrap: bool, default=FalseIf True, individual trees are fit on random subsets of the trainingdata sampled with replacement. If False, sampling without replacementis performed.",False
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for :meth:`fit`. ``None`` means 1unless in a :obj:`joblib.parallel_backend` context. ``-1`` means usingall processors. See :term:`Glossary ` for more details.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo-randomness of the selection of the featureand split values for each branching step and each tree in the forest.Pass an int for reproducible results across multiple function calls.See :term:`Glossary `.",42
,"verbose verbose: int, default=0Controls the verbosity of the tree building process.",0
,"warm_start warm_start: bool, default=FalseWhen set to ``True``, reuse the solution of the previous call to fitand add more estimators to the ensemble, otherwise, just fit a wholenew forest. See :term:`the Glossary `... versionadded:: 0.21",False


In [7]:
# 4. Predecir anomalías (-1: anomalía, 1: normal) y obtener el score
df.loc[x.index, 'anomaly'] = model.predict(x)
df.loc[x.index, 'anomaly_score'] = model.decision_function(x)

In [8]:
# 5. Visualizar resultados
print("Conteo de registros:")
print(df['anomaly'].value_counts())

Conteo de registros:
anomaly
 1.0    19574
-1.0      400
Name: count, dtype: int64


In [9]:
# Mostrar los registros detectados como anomalías
anomalies = df[df['anomaly'] == -1]
print(f"\nSe detectaron {len(anomalies)} anomalías. Muestra de las primeras:")
print(anomalies[['date', 'time'] + feature + ['anomaly_score']])


Se detectaron 400 anomalías. Muestra de las primeras:
                      date          time           fe            cu  \
222    2025-07-15 22:30:00      0.000000     0.000000      0.000000   
671    2025-07-20 14:45:00  -9999.000000 -9999.000000  -9999.000000   
851    2025-07-22 11:45:00  32994.984375  3405.180176  26545.716797   
853    2025-07-22 12:15:00  30954.634766  3593.390625  24318.937500   
1376   2025-07-27 23:00:00  33503.343750  3408.693604  28018.933594   
...                    ...           ...          ...           ...   
34058  2026-07-03 09:30:00  27692.316406  1672.609375  22980.867188   
34061  2026-07-03 10:15:00  25913.378906  2023.297607  21102.865234   
34062  2026-07-03 10:30:00  28821.832031  1959.273926  22697.435547   
34153  2026-07-04 09:15:00  27311.244141  1263.931763  22650.382812   
34168  2026-07-04 13:00:00  24930.302734  1324.868774  22021.515625   

                mo           sc  anomaly_score  
222       0.000000     0.000000      -0.115

In [10]:
# 1. Filtrar los registros normales (excluyendo el 2% de anomalías)
# Usamos el índice de la máscara de predicción
normal_indices = df[df['anomaly'] != -1].index
# 2. Generar el DataFrame limpio conservando únicamente las columnas originales
original_columns = [col for col in df.columns if col not in ['anomaly', 'anomaly_score']]
df_clean = df.loc[normal_indices, original_columns].copy()
# 3. Guardar el archivo CSV limpio
output_path = 'data/processed/intensidad_cobre_clean.csv'
df_clean.to_csv(output_path, index=False)
print(f"CSV limpio guardado en: {output_path}")
print(f"Dimensiones del archivo final: {df_clean.shape}")

CSV limpio guardado en: data/processed/intensidad_cobre_clean.csv
Dimensiones del archivo final: (19574, 7)
